# clipdinoemb

Historical experiment source from the internship project. Outputs and personal runtime metadata are cleared. This version is retained for provenance; see the root README for the maintained entry points. Colab Drive paths describe the original environment and require your own matching inputs. Source: `clipdinoemb.py`. No fresh GPU execution is claimed.

In [ ]:
# -*- coding: utf-8 -*-
"""ClipDinoEmb.ipynb

Automatically generated by Colab.

Original file is located at
    [original private Colab link omitted]
"""

from google.colab import drive

drive.mount('/content/drive')

import os


import os
import glob
import random
import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt

import torch

print("Torch Version :", torch.__version__)

print("GPU Available :", torch.cuda.is_available())

print("GPU Name :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

DATASET_PATH = "/content/drive/MyDrive/ProjectPart1"

if os.path.exists(DATASET_PATH):
    print("Dataset Found")
else:
    print("Dataset NOT Found")

room_folders = sorted(glob.glob(os.path.join(DATASET_PATH, "Room*")))

print("Total Rooms :", len(room_folders))

for room in room_folders:
    print(room)

for room in room_folders:

    images = sorted(glob.glob(os.path.join(room, "*.png")))

    print(os.path.basename(room), ":", len(images), "views")

sample_image = sorted(
    glob.glob(os.path.join(room_folders[0], "*.png"))
)[0]

img = Image.open(sample_image)

print(img.size)

plt.imshow(img)
plt.axis("off")
plt.show()

plt.figure(figsize=(20, 10))

for i, room in enumerate(room_folders):

    image_path = sorted(glob.glob(os.path.join(room, "*.png")))[0]

    img = Image.open(image_path)

    plt.subplot(4, 4, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(room))
    plt.axis("off")

plt.tight_layout()
plt.show()

for room in room_folders:
    images = sorted(glob.glob(os.path.join(room, "*.png")))

    print(os.path.basename(room), "->", len(images))

total_images = 0

for room in room_folders:

    total_images += len(
        glob.glob(os.path.join(room, "*.png"))
    )

print("Rooms :", len(room_folders))
print("Total Images :", total_images)

!pip install git+https://github.com/openai/CLIP.git

import clip
!pip install transformers timm -q
from transformers import AutoImageProcessor
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

model, preprocess = clip.load(
    "ViT-B/32",
    device=device
)

model.eval()

dinov2_processor = AutoImageProcessor.from_pretrained(
    "facebook/dinov2-base"
)

dinov2_model = AutoModel.from_pretrained(
    "facebook/dinov2-base"
).to(device)

dinov2_model.eval()

DATASET_PATH = "/content/drive/MyDrive/ProjectPart1"

import os
import glob

room_folders = sorted(
    glob.glob(os.path.join(DATASET_PATH, "Room*"))
)

sample_image_path = sorted(
    glob.glob(os.path.join(room_folders[0], "*.png"))
)[0]

print(sample_image_path)

from PIL import Image

image = Image.open(sample_image_path)

image

image_input = preprocess(image)

image_input = image_input.unsqueeze(0)

image_input = image_input.to(device)

with torch.no_grad():

    image_embedding = model.encode_image(image_input)

inputs = dinov2_processor(
    images=image,
    return_tensors="pt"
).to(device)

with torch.no_grad():

    dino_embedding = dinov2_model(**inputs)

dino_embedding = dino_embedding.last_hidden_state[:,0,:]

print(dino_embedding.shape)

print(image_embedding.shape)

image_embedding = image_embedding.cpu()

image_embedding = image_embedding.numpy()

print(image_embedding.shape)

image_embedding = image_embedding.squeeze()

print(image_embedding.shape)

print(image_embedding[:20])

embedding_norm = np.linalg.norm(image_embedding)

print(embedding_norm)

image_embedding = image_embedding / np.linalg.norm(image_embedding)

print(np.linalg.norm(image_embedding))

room_embeddings = {}

from tqdm import tqdm

for room_path in tqdm(room_folders):

    room_name = os.path.basename(room_path)

    image_paths = sorted(
        glob.glob(os.path.join(room_path, "*.png"))
    )

    room_embeddings[room_name] = []

    for image_path in image_paths:

        image = Image.open(image_path).convert("RGB")

        ############################
        # CLIP
        ############################

        clip_input = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            clip_embedding = model.encode_image(clip_input)

        clip_embedding = clip_embedding.cpu().numpy().squeeze()
        clip_embedding = clip_embedding / np.linalg.norm(clip_embedding)

        ############################
        # DINOv2
        ############################

        dino_inputs = dinov2_processor(
            images=image,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = dinov2_model(**dino_inputs)

        dino_embedding = outputs.last_hidden_state[:, 0, :]
        dino_embedding = dino_embedding.cpu().numpy().squeeze()
        dino_embedding = dino_embedding / np.linalg.norm(dino_embedding)

        ############################
        # Fusion
        ############################

        embedding = np.concatenate(
            [clip_embedding, dino_embedding]
        )

        embedding = embedding / np.linalg.norm(embedding)

        room_embeddings[room_name].append(embedding)

for room in room_embeddings:
    print(room, len(room_embeddings[room]), room_embeddings[room][0].shape)

print(room_embeddings.keys())

for room in room_embeddings:

    print(
        room,
        len(room_embeddings[room])
    )

print(
    room_embeddings["Room1"][0].shape
)

print(
    np.linalg.norm(
        room_embeddings["Room1"][0]
    )
)

all_ok = True

for room in room_embeddings:

    for emb in room_embeddings[room]:

        if emb.shape != (1280,):

            all_ok = False

print(all_ok)

random_room = random.choice(
    list(room_embeddings.keys())
)

print(random_room)

print(
    room_embeddings[random_room][3][:15]
)

count = 0

for room in room_embeddings:

    count += len(room_embeddings[room])

print(count)

total_memory = 0

for room in room_embeddings:

    for emb in room_embeddings[room]:

        total_memory += emb.nbytes

print(
    f"{total_memory/1024:.2f} KB"
)

import pickle

with open(
    "raw_fused_embeddings.pkl",
    "wb"
) as f:

    pickle.dump(room_embeddings, f)

save_path = "/content/drive/MyDrive/ProjectPart1/raw_fused_embeddings.pkl"

with open(save_path, "wb") as f:

    pickle.dump(room_embeddings, f)

with open(save_path, "rb") as f:

    room_embeddings = pickle.load(f)

image = Image.open(image_path).convert("RGB")

fused_embeddings = {}

for room in room_embeddings:

    room_embeddings[room] = [

        emb

        for emb in room_embeddings[room]

        if emb.shape == (1280,)

    ]

for room in room_embeddings:

    embeddings = np.stack(room_embeddings[room])

    fused = np.mean(
        embeddings,
        axis=0
    )

    fused_embeddings[room] = fused

for room in room_embeddings:
    print(room, "->", len(room_embeddings[room]))

for room in room_embeddings:
    print(
        room,
        "Embeddings:", len(room_embeddings[room])
    )

for room in room_embeddings:
    print(room, set([emb.shape for emb in room_embeddings[room]]))

for room in fused_embeddings:

    fused_embeddings[room] = (
        fused_embeddings[room] /
        np.linalg.norm(
            fused_embeddings[room]
        )
    )

print(
    fused_embeddings["Room1"].shape
)

print(len(fused_embeddings))

for room in fused_embeddings:

    print(
        room,
        fused_embeddings[room][:10]
    )

with open(
    "/content/drive/MyDrive/ProjectPart1/fused_clip_dino_embeddings.pkl",
    "wb"
) as f:

    pickle.dump(
        fused_embeddings,
        f
    )

with open(
    "/content/drive/MyDrive/ProjectPart1/fused_clip_dino_embeddings.pkl",
    "rb"
) as f:

    fused_embeddings = pickle.load(f)

for room in fused_embeddings:

    print(
        room,
        fused_embeddings[room].shape,
        np.linalg.norm(
            fused_embeddings[room]
        )
    )

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

room_names = sorted(fused_embeddings.keys())

print(room_names)

room_names = sorted(
    fused_embeddings.keys(),
    key=lambda x: int(x.replace("Room", ""))
)

print(room_names)

embedding_matrix = np.stack(

    [fused_embeddings[room]

     for room in room_names]

)

print(
    embedding_matrix.shape
)

similarity_matrix = cosine_similarity(

    embedding_matrix

)

print(
    similarity_matrix.shape
)

similarity_df = pd.DataFrame(

    similarity_matrix,

    index=room_names,

    columns=room_names

)

similarity_df

similarity_df = similarity_df.round(4)

similarity_df

csv_path = "/content/drive/MyDrive/ProjectPart1/multiview_similarity_matrix.csv"

similarity_df.to_csv(csv_path)

np.diag(similarity_matrix)

for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    best_match = similarities.idxmax()

    best_score = similarities.max()

    print(

        f"{room} → {best_match} ({best_score:.4f})"

    )

for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    worst_match = similarities.idxmin()

    worst_score = similarities.min()

    print(

        f"{room} → {worst_match} ({worst_score:.4f})"

    )

values = similarity_matrix[np.triu_indices(11,1)]

print("Mean :",values.mean())

print("Max :",values.max())

print("Min :",values.min())

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

sns.heatmap(similarity_df, annot=True, cmap="viridis")

plt.title(

    "Multi-View CLIP Room Similarity Matrix",

    fontsize=16

)

plt.xlabel("Rooms")

plt.ylabel("Rooms")

plt.tight_layout()
plt.show()

plt.figure(figsize=(12,10))

sns.heatmap(

    similarity_df,

    annot=True,

    cmap="viridis",

    fmt=".3f",

    square=True,

    linewidths=0.5

)

plt.title(

    "Multi-View CLIP Similarity Matrix"

)

plt.tight_layout()

plt.savefig(

"/content/drive/MyDrive/ProjectPart1/Multiview_Heatmap.png",

dpi=300

)

plt.close()

for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    best = similarities.idxmax()

    score = similarities.max()

    print(

        f"{room}  <-->  {best} : {score:.4f}"

    )

for room in room_names:

    similarities = similarity_df.loc[room].drop(room)

    worst = similarities.idxmin()

    score = similarities.min()

    print(

        f"{room}  <-->  {worst} : {score:.4f}"

    )

values = similarity_matrix[np.triu_indices(len(room_names),1)]

print(

    f"Average Similarity : {values.mean():.4f}"

)

print(

    f"Maximum Similarity : {values.max():.4f}"

)

print(

    f"Minimum Similarity : {values.min():.4f}"

)

print(

    f"Standard Deviation : {values.std():.4f}"

)

plt.figure(figsize=(8,5))

plt.hist(

    values,

    bins=15,

    edgecolor="black"

)

plt.title(

    "Distribution of Room Similarities"

)

plt.xlabel(

    "Cosine Similarity"

)

plt.ylabel(

    "Count"

)

plt.show()

pairs = []

for i in range(len(room_names)):

    for j in range(i+1,len(room_names)):

        pairs.append(

            (

                room_names[i],

                room_names[j],

                similarity_matrix[i,j]

            )

        )

pairs = sorted(

    pairs,

    key=lambda x:x[2],

    reverse=True

)

for p in pairs[:5]:

    print(p)

pairs = sorted(

    pairs,

    key=lambda x:x[2]

)

for p in pairs[:5]:

    print(p)

result_df = pd.DataFrame(

    pairs,

    columns=[

        "Room1",

        "Room2",

        "Similarity"

    ]

)

result_df.to_csv(

"/content/drive/MyDrive/ProjectPart1/Room_Pair_Similarities.csv",

index=False

)

